# 10. Robustness & Stress Testing 

Tras constatar en el notebook anterior que el modelo seleccionado (Highest Sharpe) genera un rendimiento neto sobresaliente y estadísticamente superior a una selección aleatoria de activos, la evaluación debe tornarse más rigurosa. Si bien el retorno absoluto y las métricas de eficiencia son muy favorables, la regresión CAPM reveló una fuerte exposición direccional al mercado ($\beta = 1.86$) y un alfa que, aunque económicamente masivo ($23.62\%$), roza pero no supera el estricto umbral de significación estadística del $95\%$ ($p = 0.0771$).

Con esta evidencia mixta —un motor de generación de capital muy potente pero altamente apalancado en el ciclo del mercado—, un backtest favorable no certifica la estrategia. Este notebook adopta una postura deliberadamente adversarial frente al propio sistema: en lugar de buscar nueva evidencia a favor, someteremos al candidato a producción a pruebas de estrés para descubrir bajo qué condiciones su rendimiento se degrada, se invalida estadísticamente o desaparece.

El análisis se articula en cuatro bloques. Antes de examinar la estrategia en sí, auditamos el propio proceso de búsqueda: al haber evaluado un número elevado de combinaciones de modelo, universo y frecuencia en el notebook 09, existe un riesgo estructural de sobreajuste por selección (selection bias). El Deflated Sharpe Ratio corrige explícitamente este efecto. A continuación, se somete la configuración ganadora a tres pruebas de fragilidad: sensibilidad a los hiperparámetros de entrenamiento (¿depende el resultado de la ventana de entrenamiento o el horizonte de predicción exactos que elegimos?), sensibilidad al universo de activos (¿la señal existe de verdad o es un artefacto de la composición concreta del S&P 500 elegida?), y sensibilidad al régimen de mercado (¿la estrategia funciona de forma homogénea o depende de las condiciones de volatilidad del periodo evaluado?).

## 1. Imports & Configuration

Load the libraries, notebook-level constants (prediction horizon, trading calendar, transaction cost, rebalancing frequency), and file paths used throughout this notebook.

### 1.1 Libraries

In [1]:
import sys
import pandas as pd 
import yfinance as yf
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import random
import warnings
import joblib

from pathlib import Path
from scipy.stats import spearmanr
from scipy.stats import norm, skew, kurtosis

# adds the root carpet of the project to the sys.path to allow importing modules from src
sys.path.append(str(Path.cwd().parent))

# autoreload configuration to automatically reload modules when they are modified
%load_ext autoreload
%autoreload 2

### 1.2 Notebook Configuration

In [2]:
# =============================================================================
# Experiment Configuration
# =============================================================================

PREDICTION_HORIZON = 21

TRADING_DAYS_PER_YEAR = 252

RISK_FREE_RATE = 0.0

REBALANCING_FREQUENCY = 21

BASE_TRANSACTION_COST = 0.0015 

FINAL_CANDIDATE = {
    "role": "Highest Sharpe",
    "model": "Random Forest",
    "portfolio": "long_only_top_10_signal_weight",
    "frequency_days": 21,
}

### 1.3 Paths & Global Parameters

In [3]:
# =============================================================================
# Paths
# =============================================================================

BENCHMARK_RETURNS_PATH = "../data/portfolio_results/sp500_benchmark_returns.parquet"

NET_RETURNS_PATH = "../data/portfolio_results/net_portfolio_returns.parquet"

SENSITIVITY_METRICS_PATH = "../data/portfolio_results/rebalancing_frequency_sensitivity_metrics.parquet"

## 2. Data Loading

In [4]:
# =============================================================================
# Load Extended Price Data
# =============================================================================

prices = pd.read_parquet("../data/raw/sp500_prices_extended.parquet")

# =============================================================================
# Compute Daily Asset Returns
# =============================================================================

adj_close = prices["Adj Close"]

simple_returns = adj_close.pct_change()

# =============================================================================
# Load Net Portfolio Returns
# =============================================================================

net_returns = pd.read_parquet(
    "../data/portfolio_results/net_portfolio_returns.parquet"
)


# =============================================================================
# Load Benchmark Returns
# =============================================================================

sp500_benchmark_returns = pd.read_parquet(
    "../data/portfolio_results/sp500_benchmark_returns.parquet"
)


## 3. Deflated Sharpe Ratio & Corrección por Multiple Testing

El hallazgo de un ratio Sharpe elevado en un backtest carece de validez si forma parte de una búsqueda masiva de especificaciones. Al explorar múltiples combinaciones de modelos, universos y ponderaciones, la probabilidad de identificar una estrategia "ganadora" por puro azar aumenta de forma exponencial (selection bias o p-hacking). Este bloque implementa el Deflated Sharpe Ratio (DSR) (Bailey & López de Prado, 2014), un marco analítico que corrige el Sharpe observado ajustándolo por la no normalidad de los retornos (asimetría y curtosis), el tamaño de la muestra y, de forma crítica, el número total de ensayos (trials) independientes evaluados durante el proceso de investigación.

### 3.1 Trial Count & Cross-Sectional Sharpe Variance

Para calcular estadísticamente el umbral de Sharpe esperado por puro azar, es imperativo cuantificar con rigor el espacio de búsqueda (trial count, $N$) y la dispersión de los resultados. A nivel metodológico, se adopta una distinción clave: separar las hipótesis estructurales netas (combinaciones únicas de modelo y esquema de ponderación evaluadas a la frecuencia base de 21 días) de las variaciones puramente operativas (frecuencias de rebalanceo alternativas como 5, 10 o 42 días, que replican la misma señal bajo otro horizonte de ejecución). El código contrasta así dos escenarios: uno conservador (que computa toda la rejilla de experimentos) y uno defendible (que aísla las señales distintas a 21 días). Paralelamente, se calcula la varianza cross-sectional de los Sharpe para dimensionar la dispersión del universo analizado.

In [5]:
# =============================================================================
# 3.1 Trial Count & Cross-Sectional Sharpe Variance
# =============================================================================

df_full_grid = pd.read_parquet(
    "../data/portfolio_results/rebalancing_frequency_sensitivity_metrics.parquet"
)

# We keep the chosen operational frequency (21 days) to isolate "how many
# distinct signal/weighting hypotheses we have tested" from the effect of the
# execution frequency, which is not a new hypothesis but an execution variant
# of the same signal.
df_trials = df_full_grid[df_full_grid["frequency_days"] == 21].copy()

n_trials_distinct_signals = df_trials[["model", "portfolio"]].drop_duplicates().shape[0]
n_trials_full_grid = len(df_full_grid)
sharpe_variance = df_trials["Sharpe"].var(ddof=1)

print("=" * 80)
print("TRIAL COUNT & CROSS-SECTIONAL SHARPE VARIANCE")
print("=" * 80)
print(f"✓ Trials (model × scheme, at 21 days):      {n_trials_distinct_signals}")
print(f"✓ Trials (full grid, incl. frequencies):    {n_trials_full_grid}")
print(f"✓ Cross-sectional Sharpe variance:          {sharpe_variance:.5f}")

TRIAL COUNT & CROSS-SECTIONAL SHARPE VARIANCE
✓ Trials (model × scheme, at 21 days):      42
✓ Trials (full grid, incl. frequencies):    210
✓ Cross-sectional Sharpe variance:          0.22109


### 3.2 Deflated Sharpe Ratio — Winning Candidate

Una vez parametrizado el espacio de búsqueda y la varianza del sistema, se audita formalmente la robustez estadística de la configuración ganadora (Random Forest, Top 10 Signal Weight, 21 días). El algoritmo calcula el Sharpe máximo esperado por puro azar (Expected Maximum Sharpe) en función de los $N$ ensayos y evalúa el Sharpe observado de la estrategia —incorporando estrictamente las correcciones por asimetría (skewness) y curtosis no excesiva de su serie diaria de retornos netos— mediante el Probabilistic Sharpe Ratio (PSR). El valor resultante (deflated PSR) arroja la probabilidad matemática de que el verdadero rendimiento ajustado por riesgo de la estrategia supere al mejor resultado atribuible al azar, exigiendo un umbral de confianza superior al $95\%$ ($> 0.95$) para descartar definitivamente que el éxito sea un artefacto de la optimización.

In [6]:
# =============================================================================
# 3.2 Deflated Sharpe Ratio — Winning Candidate
# =============================================================================

EULER_MASCHERONI = 0.5772156649

from src.portfolio.utils import probabilistic_sharpe_ratio, expected_max_sharpe

net_returns = pd.read_parquet("../data/portfolio_results/net_portfolio_returns.parquet")

winning_mask = (
    (net_returns["model"] == "Random Forest")
    & (net_returns["portfolio"] == "long_only_top_10_signal_weight")
)
winning_daily_returns = net_returns.loc[winning_mask, "net_return_base"].dropna()

metric_mask = (
    (df_full_grid["model"] == "Random Forest")
    & (df_full_grid["portfolio"] == "long_only_top_10_signal_weight")
    & (df_full_grid["frequency_days"] == 21)
)
observed_sharpe = df_full_grid.loc[metric_mask, "Sharpe"].values[0]
ret_skew = skew(winning_daily_returns)
ret_kurtosis = kurtosis(winning_daily_returns, fisher=False)  # Raw kurtosis, not excess kurtosis
n_obs = len(winning_daily_returns)

dsr_results = {}
for label, n_trials in [
    ("Conservative (full grid)", n_trials_full_grid),
    ("Defensible (distinct signals, 21d)", n_trials_distinct_signals),
]:
    sr_benchmark = expected_max_sharpe(sharpe_variance, n_trials)
    psr = probabilistic_sharpe_ratio(observed_sharpe, sr_benchmark, n_obs, ret_skew, ret_kurtosis)
    dsr_results[label] = {
        "n_trials": n_trials,
        "expected_max_sharpe_by_chance": sr_benchmark,
        "deflated_psr": psr,
    }

df_dsr = pd.DataFrame(dsr_results).T

print("=" * 90)
print("DEFLATED SHARPE RATIO (RANDOM FOREST — TOP 10 SIGNAL WEIGHT — 21D)")
print("=" * 90)
print(f"✓ Observed Sharpe:             {observed_sharpe:.3f}")
print(f"✓ Daily Skewness / Kurtosis:   {ret_skew:.3f} / {ret_kurtosis:.3f}")
print(f"✓ Observations (n):            {n_obs}")
print("-" * 90)
print(df_dsr.to_string(formatters={
    "n_trials": lambda x: f"{int(x)}",
    "expected_max_sharpe_by_chance": lambda x: f"{x:.3f}",
    "deflated_psr": lambda x: f"{x:.4f}",
}))
print("-" * 90)
print("Note: deflated_psr is the probability that the strategy's true Sharpe ratio")
print("exceeds the maximum expected Sharpe by chance after testing n_trials configurations.")
print("A value > 0.95 rejects the hypothesis that the result is a search artifact.")

DEFLATED SHARPE RATIO (RANDOM FOREST — TOP 10 SIGNAL WEIGHT — 21D)
✓ Observed Sharpe:             2.474
✓ Daily Skewness / Kurtosis:   0.151 / 9.002
✓ Observations (n):            392
------------------------------------------------------------------------------------------
                                   n_trials expected_max_sharpe_by_chance deflated_psr
Conservative (full grid)                210                         1.308       1.0000
Defensible (distinct signals, 21d)       42                         1.039       1.0000
------------------------------------------------------------------------------------------
Note: deflated_psr is the probability that the strategy's true Sharpe ratio
exceeds the maximum expected Sharpe by chance after testing n_trials configurations.
A value > 0.95 rejects the hypothesis that the result is a search artifact.


Los resultados del análisis de *Deflated Sharpe Ratio* (DSR) confirman de manera categórica que el rendimiento ajustado por riesgo de la estrategia ganadora (*Random Forest, Top 10 Signal Weight, 21d*) no constituye un artefacto estadístico derivado del sesgo de selección o de pruebas múltiples (*multiple testing*).

Al evaluar el espacio de búsqueda bajo dos escenarios metodológicos distintos, el umbral de eficiencia que cabría esperar alcanzar por puro azar se mantiene sustancialmente por debajo del rendimiento observado:

* **Escenario defensivo ($N = 42$ señales distintas a 21 días):** Sitúa el Sharpe máximo esperado por azar en **1.039**.
* **Escenario conservador ($N = 210$ ensayos totales de la rejilla):** Eleva el listón del azar hasta un Sharpe de **1.308** debido a la mayor amplitud del espacio explorado.

Frente a estos límites teóricos, el Sharpe observado de **2.474** se mantiene en un plano netamente superior. Este diferencial de eficiencia resiste de forma rigurosa la penalización impuesta por las características distribucionales de la serie temporal: a pesar de registrar una curtosis elevada de **9.002** (reflejo de colas pesadas inherentes a la concentración en deciles del modelo predictivo) y un tamaño de muestra de **$392$ observaciones** *out-of-sample*, el *Deflated Probabilistic Sharpe Ratio* alcanza un valor de **1.0000** en ambas especificaciones.

Esta probabilidad del 100% permite rechazar con absoluta certeza estadística la hipótesis nula de que la rentabilidad de la cartera provenga de la suerte o del *p-hacking* acumulado al barrer la rejilla de modelos. El alfa y la eficiencia del candidato seleccionado poseen un fundamento informacional genuino, validando su transición hacia las siguientes fases de pruebas de estrés estructural y robustez en el notebook 10.

## 4. Sensibilidad a Hiperparámetros

En qué consiste: reentrenar la misma arquitectura ganadora (Random Forest + Top 10 Signal Weight, para no multiplicar el coste computacional probando las tres arquitecturas) con una rejilla pequeña de ventanas de entrenamiento y horizontes de predicción, y comparar las métricas resultantes con el mismo patrón de tabla que usasteis en la sensibilidad de frecuencia del notebook 09.

In [7]:
# =============================================================================
# 4.0 Configuration Constants
# =============================================================================

BASE_TRANSACTION_COST = 0.0015     # 15 bps, igual que en notebook 09
REBALANCING_DAYS_FIXED = 21        # fijo: aísla el efecto del horizonte de predicción
                                    # del efecto de frecuencia de rebalanceo (ya medido en 09§5)
MODEL_LABEL = "Random Forest"
PORTFOLIO_LABEL = "long_only_top_10_signal_weight"

In [8]:
from sklearn.ensemble import RandomForestRegressor
from src.portfolio.constraints import apply_turnover_constraint_with_drift
from src.models.training import run_cpcv_training
from src.models.utils import CombinatorialPurgedCV
import json


df_rank = pd.read_parquet(
    "../data/preprocessed/df_final_rank.parquet"
)

FEATURES_RANK = [
    "momentum_12_1_win_rank", 
    "upside_volatility_win_rank", 
    "log10_amihud_win_rank",
]

# 1. Cargar el JSON con las configuraciones
with open("../data/model_results/final_model/final_model_configurations.json", "r") as f:
    model_configs = json.load(f)

# 2. Extraer el bloque específico del Random Forest
rf_config = model_configs["Random Forest"]

# 3. Construir el diccionario plano que necesita scikit-learn
best_params_rf_rank = rf_config["hyperparameters"].copy()
best_params_rf_rank["random_state"] = rf_config["random_state"]
# n_jobs se forzará a -1 dentro de run_pipeline, no hace falta añadirlo aquí.

def run_pipeline(
    train_window_years: int,
    prediction_horizon_days: int,
    df_features: pd.DataFrame = df_rank,
    returns_df: pd.DataFrame = simple_returns,
    features_cols: list = FEATURES_RANK,
    rf_params: dict = best_params_rf_rank,
    rebalancing_days: int = REBALANCING_DAYS_FIXED,
    transaction_cost: float = BASE_TRANSACTION_COST,
) -> pd.Series:
    """
    Pipeline end-to-end: target dinámico, entrenamiento, cartera Top 10%
    Signal Weight, restricciones y backtest neto.

    rebalancing_days se mantiene FIJO (no ligado a prediction_horizon_days)
    para no mezclar el efecto del horizonte de predicción con el de la
    frecuencia de rebalanceo.
    """
    print(
        f"--- Pipeline: Train {train_window_years}y | Horizon {prediction_horizon_days}d "
        f"| Rebal {rebalancing_days}d ---"
    )

    # =========================================================================
    # 1. TARGET DINÁMICO
    # =========================================================================
    print("1. Calculando Target Compuesto...")
    forward_returns = (
        (1 + returns_df).rolling(window=prediction_horizon_days).apply(np.prod, raw=True) - 1
    ).shift(-prediction_horizon_days)

    # Renombrar explícitamente los niveles del índice para que coincidan con df_features
    target_series = forward_returns.stack().rename_axis(["date", "ticker"]).rename("target")
    
    # Ahora el join funcionará correctamente
    df_model = df_features[features_cols].join(target_series).dropna()

    # -------------------------------------------------------------------
    # FIX — truncar el histórico ANTES de generar los splits, para que
    # train_window_years de verdad limite lo que ve el modelo.
    # -------------------------------------------------------------------
    dates_available = df_model.index.get_level_values("date").unique().sort_values()
    lookback_start = (
        dates_available.max()
        - pd.DateOffset(years=train_window_years)
        - pd.Timedelta(days=prediction_horizon_days * 3)  # margen de purga/embargo del primer fold
    )
    df_model = df_model.loc[df_model.index.get_level_values("date") >= lookback_start]

    actual_years_span = (
        df_model.index.get_level_values("date").max()
        - df_model.index.get_level_values("date").min()
    ).days / 365.25
    print(f"   Histórico efectivo tras truncar: {actual_years_span:.2f} años (objetivo: {train_window_years})")
    assert actual_years_span <= train_window_years + 0.5, (
        "El truncado no ha reducido el histórico como se esperaba — revisa "
        "dates_available o el formato del índice de df_model."
    )

    X_subset = df_model[features_cols]
    y_subset = df_model["target"]

    # =========================================================================
    # 2. ENTRENAMIENTO Y PREDICCIÓN
    # =========================================================================
    print("2. Entrenando Random Forest y generando OOS predictions...")

    rf_params_run = rf_params.copy()
    # Importante: Como tu función run_cpcv_training ya usa Parallel(n_jobs=4), 
    # evitamos que Random Forest haga su propio paralelismo interno para que no explote la RAM.
    rf_params_run["n_jobs"] = 1 

    # A) Instanciar el creador de splits adaptando la purga al horizonte predictivo actual
    cpcv = CombinatorialPurgedCV(
        n_blocks=7,
        k_validation=2,
        purge_window=prediction_horizon_days,  # La purga se adapta a 21 o 63 días
        embargo_window=11,
    )

    # B) Generar los índices (los splits)
    dynamic_splits = list(cpcv.split(X_subset))

    # C) Pasar los splits a TU función de entrenamiento
    oos_preds, _, _ = run_cpcv_training(
        model_cls=RandomForestRegressor,
        model_params=rf_params_run,
        X=X_subset,
        y=y_subset,
        splits=list(cpcv.split(X_subset)),
        rank_method="spearman",
    )

    # FIX: Extraer específicamente la columna de predicción del DataFrame de 3 columnas
    if isinstance(oos_preds, pd.Series):
        oos_predictions = oos_preds.to_frame("rf_prediction")
    elif isinstance(oos_preds, pd.DataFrame):
        if "prediction" in oos_preds.columns:
            oos_predictions = oos_preds[["prediction"]].rename(columns={"prediction": "rf_prediction"})
        else:
            # Fallback por si la columna se llama diferente, coge la última (suele ser la predicción)
            pred_col = oos_preds.columns[-1]
            oos_predictions = oos_preds[[pred_col]].rename(columns={pred_col: "rf_prediction"})

    # Check de seguridad para evitar que pase vacío al Paso 4
    if len(oos_predictions) == 0:
        raise ValueError("oos_predictions está vacío. Falló la generación de predicciones.")

    # =========================================================================
    # 3. CARTERA TOP 10% SIGNAL WEIGHT
    # =========================================================================
    print("3. Construyendo Cartera Top 10%...")

    oos_predictions["rank_pct"] = (
        oos_predictions["rf_prediction"].groupby(level="date").rank(method="first", pct=True)
    )
    top_10_mask = oos_predictions["rank_pct"] >= 0.90
    selected_preds = oos_predictions.loc[top_10_mask].copy()

    selected_preds["signal_strength"] = (
        selected_preds["rf_prediction"].groupby(level="date").rank(method="first", pct=True)
    ) ** 1.0

    selected_preds["weight"] = (
        selected_preds["signal_strength"]
        / selected_preds.groupby(level="date")["signal_strength"].transform("sum")
    )

    portfolio_weights = selected_preds[["weight"]].reset_index()

    # FIX — la función ya aplica min_weight y max_weight internamente (su Paso 2);
    # el bucle manual anterior los estaba renormalizando dos veces. Se elimina.
    # FIX — se añaden las columnas que la función necesita para su groupby interno.
    portfolio_weights["model"] = MODEL_LABEL
    portfolio_weights["portfolio"] = PORTFOLIO_LABEL
    
    print(f"   -> Activos seleccionados para cartera: {len(portfolio_weights)} filas.")
    if len(portfolio_weights) == 0:
        raise ValueError("La selección de cartera está vacía. Revisa las predicciones OOS.")

    # =========================================================================
    # 4. TURNOVER CONSTRAINT + BOX CONSTRAINTS (una sola llamada)
    # =========================================================================
    print("4. Aplicando Restricciones de Peso y Turnover...")

    rebalance_weights, _ = apply_turnover_constraint_with_drift(
        portfolio_weights,
        returns_df,
        rebalancing_days=rebalancing_days,
        max_turnover_design=0.25,
        min_weight=0.005,
        max_weight=0.05,
    )
    # Se descarta a propósito el segundo valor (daily_weights): mantiene el peso
    # constante entre rebalanceos (sin drift), y usarlo para calcular retornos
    # asumiría un rebalanceo diario gratuito que nunca se está cobrando. El drift
    # se simula manualmente abajo, con el mismo patrón validado en notebook 09 §7.1.

    rebalance_wide = (
        rebalance_weights.pivot(index="date", columns="ticker", values="weight")
        .reindex(columns=returns_df.columns)  # <-- Alinea los 83 activos a los 496 totales
        .fillna(0.0)
        .sort_index()
    )
    rebalance_dates = rebalance_wide.index

    # =========================================================================
    # 5. TURNOVER REALIZADO Y COSTES — SOLO EN FECHAS DE REBALANCEO
    # =========================================================================
    print("5. Calculando Turnover Real (con drift) y Costes...")

    turnover_records = [
        {"date": rebalance_dates[0], "turnover": 0.5 * rebalance_wide.loc[rebalance_dates[0]].abs().sum()}
    ]
    for i in range(1, len(rebalance_dates)):
        prev_reb, curr_reb = rebalance_dates[i - 1], rebalance_dates[i]
        prev_w = rebalance_wide.loc[prev_reb]

        window_rets = returns_df.loc[(returns_df.index > prev_reb) & (returns_df.index <= curr_reb)]
        compounded = ((1.0 + window_rets).prod() - 1.0).reindex(prev_w.index).fillna(0.0)

        drifted = prev_w * (1.0 + compounded)
        drifted_norm = drifted / drifted.sum() if drifted.sum() > 0 else prev_w

        curr_w = rebalance_wide.loc[curr_reb]
        turnover_records.append({"date": curr_reb, "turnover": 0.5 * (curr_w - drifted_norm).abs().sum()})

    df_turnover_schedule = pd.DataFrame(turnover_records).set_index("date")
    df_turnover_schedule["transaction_cost_pct"] = df_turnover_schedule["turnover"] * transaction_cost

    # =========================================================================
    # 6. EJECUCIÓN DIARIA CON DRIFT (T+1) Y RETORNOS NETOS
    # =========================================================================
    print("6. Simulando Ejecución (T+1, con drift) y calculando Retornos Netos...")

    all_trading_dates = returns_df.index
    executed_weights_list = []

    for i in range(len(rebalance_dates)):
        reb_date_t = rebalance_dates[i]
        reb_idx = all_trading_dates.get_loc(reb_date_t)
        if reb_idx + 1 >= len(all_trading_dates):
            break

        exec_start_date = all_trading_dates[reb_idx + 1]
        if i + 1 < len(rebalance_dates):
            window_returns = returns_df.loc[exec_start_date:rebalance_dates[i + 1]]
        else:
            window_returns = returns_df.loc[exec_start_date:]

        w_current = rebalance_wide.loc[reb_date_t].values
        for date, ret in window_returns.iterrows():
            executed_weights_list.append(pd.Series(w_current, index=returns_df.columns, name=date))
            w_gross = w_current * (1.0 + ret.values)
            total_gross = w_gross.sum()
            w_current = w_gross / total_gross if total_gross != 0.0 else w_gross

    executed_matrix = pd.DataFrame(executed_weights_list)

    gross_returns = (
        (executed_matrix.shift(1) * returns_df.reindex(executed_matrix.index)).sum(axis=1).dropna()
    )

    daily_costs_pct = pd.Series(0.0, index=gross_returns.index)
    cost_dates = gross_returns.index.intersection(df_turnover_schedule.index)
    daily_costs_pct.loc[cost_dates] = df_turnover_schedule.loc[cost_dates, "transaction_cost_pct"]

    print("--- Pipeline Completado ---")

    oos_start_date = "2025-01-15"
    # Aislar y devolver EXCLUSIVAMENTE el periodo Out-Of-Sample
    net_returns_oos = net_returns.loc[oos_start_date:]
    
    return net_returns_oos


net_returns_base = run_pipeline(train_window_years=5, prediction_horizon_days=21)

# Comprobar la salida
print(net_returns_base.head())

c:\Users\dppec\OneDrive\Escritorio\PROYECTOS DE PYTHON\quant-portfolio-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Pipeline: Train 5y | Horizon 21d | Rebal 21d ---
1. Calculando Target Compuesto...
   Histórico efectivo tras truncar: 5.17 años (objetivo: 5)
2. Entrenando Random Forest y generando OOS predictions...
3. Construyendo Cartera Top 10%...
   -> Activos seleccionados para cartera: 379140 filas.
4. Aplicando Restricciones de Peso y Turnover...
5. Calculando Turnover Real (con drift) y Costes...
6. Simulando Ejecución (T+1, con drift) y calculando Retornos Netos...
--- Pipeline Completado ---
           date          model                        portfolio  gross_return  \
2026 2025-03-27  Random Forest  long_only_top_30_maximum_sharpe     -0.001716   
2027 2025-03-27  Random Forest     long_only_top_30_risk_parity      0.002040   
2028 2025-03-27  Random Forest   long_only_top_30_signal_weight     -0.013223   
2029 2025-03-27  Random Forest          long_short_equal_weight     -0.009974   
2030 2025-03-27          Ridge           long_only_equal_weight     -0.006801   

      transactio

In [9]:
from src.portfolio.metrics import calculate_sharpe

sharpe_base = calculate_sharpe(net_returns_base)
print(f"Sharpe Ratio Base: {sharpe_base:.3f}")

TypeError: unsupported operand type(s) for +: 'float' and 'DatetimeArray'